# GeoAI data pipeline: Earth Engine → BigQuery → DuckDB / MotherDuck

A teaching notebook showing where DuckDB and MotherDuck fit in a GeoAI workflow.

**Core idea:** keep big imagery in Earth Engine (or COG/Zarr). Reduce it to *vector and tabular* results
(zonal statistics, point samples, features), then use SQL (DuckDB locally, MotherDuck in the cloud) for
joins, aggregation, QA and sharing.

```
 Earth Engine (rasters)         BigQuery / Databricks (optional)         DuckDB / MotherDuck
 reduceRegions / sampleRegions → Export.table.toBigQuery / Delta tables → bronze → silver → gold (GeoParquet)
```

**What is tested and what is not**

| Section | Status |
|---|---|
| 3-6 (MotherDuck, spatial SQL, gold table) | SQL tested live on a MotherDuck account (Oct 2026) |
| 2 (Earth Engine export) | Reference code only. Needs your own Earth Engine + Google Cloud project. Check current API docs |
| 4c (BigQuery / Databricks attach) | Optional, untested. Depends on community extension support |
| Zonal stats in section 4 | **Synthetic stand-in** (clearly labeled `SYNTHETIC_DEMO`), so the notebook runs without Earth Engine |

## 0. Setup

In [ ]:
# !pip install duckdb geopandas matplotlib pyarrow
import os, duckdb

# Never hard-code your token. Set it in your shell or Colab secrets first:
#   export motherduck_token="..."   (create one in the MotherDuck UI: Settings > Tokens)
assert os.environ.get("motherduck_token"), "Set the motherduck_token environment variable first"

con = duckdb.connect("md:")          # connects to MotherDuck with the token above
print(con.sql("SELECT current_database(), version()").fetchall())

## 1. Where each tool fits

| Task | Best tool | Why |
|---|---|---|
| Multi-terabyte raster analysis, compositing, indices | Earth Engine | Server-side raster compute |
| Large warehouse tables, sharing with a GCP team | BigQuery | Native Earth Engine export path, `GEOGRAPHY` type |
| Lakehouse / Delta tables | Databricks (free edition) | Unity Catalog, notebooks |
| Fast SQL on vectors, tables, GeoParquet | DuckDB | In-process, spatial extension, reads Parquet/CSV from URLs |
| Same, but shared, persistent and cloud-hosted | MotherDuck | DuckDB in the cloud, hybrid local/cloud queries |

Rule of thumb: **raster pixels stay in raster tools; summaries come to SQL.**

## 2. Earth Engine: reduce imagery to a table (reference, not executed)

Run this in your own Earth Engine environment. It computes monthly county-level mean NDVI and
precipitation. Exporting to BigQuery requires a Cloud project with the BigQuery API enabled.
Check the current Earth Engine docs ("BigQuery integrations") because export parameters change.

In [ ]:
# --- REFERENCE ONLY (requires `earthengine authenticate` and a GCP project) ---
# import ee; ee.Initialize(project="YOUR_PROJECT")
#
# counties = (ee.FeatureCollection("TIGER/2018/Counties")
#               .filter(ee.Filter.eq("STATEFP", "48")))        # Texas
#
# def monthly_stats(month_start):
#     start = ee.Date(month_start); end = start.advance(1, "month")
#     ndvi = (ee.ImageCollection("MODIS/061/MOD13Q1").filterDate(start, end)
#               .select("NDVI").mean().multiply(0.0001).rename("mean_ndvi"))
#     pr = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate(start, end)
#             .select("precipitation").sum().rename("precip_mm"))
#     return (ndvi.addBands(pr).reduceRegions(counties, ee.Reducer.mean(), scale=5000)
#               .map(lambda f: f.set("obs_month", start.format("YYYY-MM-dd"))))
#
# months = ee.List.sequence(0, 23).map(lambda i: ee.Date("2023-01-01").advance(i, "month"))
# table = ee.FeatureCollection(months.map(monthly_stats)).flatten()
#
# task = ee.batch.Export.table.toBigQuery(
#     collection=table, description="tx_county_monthly_stats",
#     table="YOUR_PROJECT.geoai_demo.tx_county_monthly_stats")
# task.start()

## 3. Create a demo database in MotherDuck

Everything below lives in its own database so it is easy to delete at the end.

In [ ]:
con.sql("CREATE DATABASE IF NOT EXISTS geoai_pipeline_demo")
con.sql("USE geoai_pipeline_demo")
con.sql("SELECT current_database()").show()

## 4. Bronze layer: raw inputs

**4a. County polygons.** `ST_Read` (from DuckDB's `spatial` extension) reads GeoJSON straight from a URL.
Texas FIPS codes start with `48`.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE bronze_counties AS
SELECT id AS fips, geom
FROM ST_Read('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json')
WHERE id LIKE '48%'
""")
con.sql("SELECT count(*) AS texas_counties FROM bronze_counties").show()   # expect 254

**4b. Zonal statistics.** This cell builds a **synthetic stand-in** with the same shape as the Earth Engine output
from section 2 (`fips, obs_month, mean_ndvi, precip_mm`). The values follow a simple west-to-east gradient and a
seasonal cycle. They are **not real measurements**. Replace this cell with your real export when ready.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE bronze_zonal_stats AS
WITH c AS (
  SELECT fips, ST_X(ST_Centroid(geom)) AS lon FROM bronze_counties
)
SELECT c.fips,
       make_date(y.yr, mo.month, 1) AS obs_month,
       round(0.25 + 0.35*((c.lon+106.6)/13.0) * (0.6+0.4*sin((mo.month-3)*pi()/6)) + 0.02*(y.yr-2023), 4) AS mean_ndvi,
       round(greatest(5, 30 + 60*((c.lon+106.6)/13.0) + 25*sin((mo.month-5)*pi()/6)), 1) AS precip_mm,
       'SYNTHETIC_DEMO' AS source
FROM c,
     (SELECT unnest([2023, 2024]) AS yr) y,
     (SELECT unnest(range(1, 13)) AS month) mo
""")
con.sql("SELECT count(*) AS rows FROM bronze_zonal_stats").show()           # expect 6096 (254 x 24)
con.sql("SELECT * FROM bronze_zonal_stats LIMIT 5").show()

**4c. (Optional, untested) Pull the real table from BigQuery.**
DuckDB has a community BigQuery extension, and MotherDuck documents a BigQuery integration.
Support for community extensions on MotherDuck's servers may differ from local DuckDB, so the safe pattern is:
attach BigQuery from a **local** DuckDB, then write the result into MotherDuck. Test with a small table first.

In [ ]:
# --- OPTIONAL / UNTESTED ---
# local = duckdb.connect()
# local.sql("INSTALL bigquery FROM community; LOAD bigquery;")
# local.sql("ATTACH 'project=YOUR_PROJECT' AS bq (TYPE bigquery, READ_ONLY)")
# df = local.sql("SELECT * FROM bq.geoai_demo.tx_county_monthly_stats").arrow()
# con.sql("CREATE OR REPLACE TABLE bronze_zonal_stats AS SELECT * FROM df")

## 5. Silver layer: clean and validate

Typical silver steps: fix types, remove impossible values, add quality flags. Make checks explicit so
students can see them.

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW silver_zonal_stats AS
SELECT *,
       (mean_ndvi BETWEEN -0.2 AND 1.0) AS ndvi_valid,
       (precip_mm >= 0)                 AS precip_valid
FROM bronze_zonal_stats
""")
con.sql("""
SELECT count(*) FILTER (WHERE NOT ndvi_valid)   AS bad_ndvi,
       count(*) FILTER (WHERE NOT precip_valid) AS bad_precip,
       count(DISTINCT fips)                     AS counties
FROM silver_zonal_stats
""").show()

## 6. Gold layer: county summary + spatial queries

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE gold_county_summary AS
SELECT z.fips,
       round(avg(mean_ndvi), 4)               AS avg_ndvi,
       round(sum(precip_mm), 1)               AS total_precip_mm,
       round(corr(mean_ndvi, precip_mm), 3)   AS ndvi_precip_corr,
       any_value(c.geom)                      AS geom
FROM silver_zonal_stats z
JOIN bronze_counties c USING (fips)
WHERE ndvi_valid AND precip_valid
GROUP BY z.fips
""")
con.sql("SELECT count(*) AS rows FROM gold_county_summary").show()

**Point-in-polygon:** which county contains Kingsville, TX? Note that spatial functions take **(x, y) = (lon, lat)**.

In [ ]:
con.sql("""
SELECT fips, avg_ndvi, total_precip_mm, ndvi_precip_corr
FROM gold_county_summary
WHERE ST_Contains(geom, ST_Point(-97.8561, 27.5159))
""").show()          # expect fips 48273 (Kleberg County)

**Common gotcha: spheroid functions use (lat, lon) order.** `ST_Distance_Spheroid` expects latitude first,
so passing (lon, lat) silently returns `NaN`. Compare:

In [ ]:
con.sql("""
SELECT ST_Distance_Spheroid(ST_Point(-97.5, 27.5), ST_Point(-98.5, 29.4)) AS wrong_order_lonlat,
       ST_Distance_Spheroid(ST_Point(27.5, -97.5), ST_Point(29.4, -98.5)) AS meters_latlon
""").show()          # first is NaN, second is about 232 km

## 7. Export and map (local)

In [ ]:
import geopandas as gpd, matplotlib.pyplot as plt

# Pull the gold table into a GeoDataFrame (geometry as WKB)
df = con.sql("SELECT fips, avg_ndvi, total_precip_mm, ST_AsWKB(geom) AS wkb FROM gold_county_summary").df()
gdf = gpd.GeoDataFrame(df.drop(columns="wkb"),
                       geometry=gpd.GeoSeries.from_wkb(df["wkb"].apply(bytes)), crs="EPSG:4326")

gdf.to_parquet("tx_county_summary.parquet")      # GeoParquet, shareable with any GIS tool
gdf.plot(column="avg_ndvi", cmap="YlGn", legend=True, figsize=(7, 7))
plt.title("Mean NDVI by Texas county (SYNTHETIC demo data)"); plt.axis("off"); plt.show()

## 8. Exercises

1. Replace section 4b with a real Earth Engine export for one county and compare against the synthetic pattern.
2. Add a `silver` check that flags counties with fewer than 24 monthly observations.
3. Use `ST_Intersects` with a hand-drawn study-area polygon and summarize only the counties it touches.
4. Share the gold table with a colleague using a MotherDuck share (`CREATE SHARE`).
5. Time the same aggregation locally on a GeoParquet file versus in MotherDuck. When does each win?

### Using Claude with the MotherDuck connector
Once the demo database exists, you can ask Claude things like:
- "List the tables in `geoai_pipeline_demo` and describe their columns."
- "Which five Texas counties have the highest `ndvi_precip_corr`?"
- "Write a query for monthly NDVI anomalies per county."

Ask Claude to review queries before it runs anything that writes data.

## 9. Clean up

In [ ]:
# Uncomment to remove everything created by this notebook
# con.sql("USE my_db")
# con.sql("DROP DATABASE geoai_pipeline_demo")